# Colab Evaluation Test - xlam

- 입력: `q` (문자열 JSON)
- 정답: `a_w1`~`a_w8` (없으면 `a` JSON에서 추출)
- 출력: 전체 정확도, 모터별 정확도, 평균 절대 오차(MAE), 파일/구간별 결

In [1]:
# 필요 패키지
!pip -q install transformers==5.0.0 accelerate==1.13.0 peft==0.18.1 bitsandbytes==0.49.2 scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.5 MB/s eta 0:00:00


In [ ]:
import os
import re
import ast
import json
import glob
import numpy as np
import pandas as pd
import transformers
import peft
import datasets
import accelerate
from tqdm.auto import tqdm
import bitsandbytes

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, AutoPeftModelForCausalLM, PeftConfig

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"PEFT version: {peft.__version__}")
print(f"Datasets version: {datasets.__version__}")
print(f"Accelerate version: {accelerate.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print("Installation successful and modules imported.")

PyTorch version: 2.10.0+cu128
Transformers version: 5.0.0
PEFT version: 0.18.1
Datasets version: 4.0.0
Accelerate version: 1.13.0
bitsandbytes: 0.49.2
Installation successful and modules imported.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ===== 경로 설정 =====
main_name = '20260415_xLAM-2-8b-fc-r_10486-row_20percent-action_min_4motors'
HOME_PATH = '/content/drive/MyDrive/TANGO2 - SDF/'

FINETUNED_MODEL_PATH = f'{HOME_PATH}모델/{main_name}/best_model'
EVAL_DATA_PATH = f'/content/drive/MyDrive/TANGO2 - SDF/데이터/최기형_양배추_20260211_20260402_v3_augmentation/원천데이터/10_qa_balanced_min/balanced_min_train_by_20_test.csv'
OUTPUT_FILE_PATH = f'{HOME_PATH}데이터/최기형_양배추_20260211_20260402_v3_augmentation/추론데이터/20260415_xLAM-2-8b-fc-r_10486-row_20percent-action_min_4motors_predict_balanced_min_train_by_20_test.csv'
motor_cols = [f'w{i}' for i in range(1, 9)]

os.makedirs(os.path.dirname(OUTPUT_FILE_PATH), exist_ok=True)

In [ ]:
from transformers import BitsAndBytesConfig

# ===== 모델 로드 =====
tokenizer = AutoTokenizer.from_pretrained(FINETUNED_MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# adapter config 읽기
peft_config = PeftConfig.from_pretrained(FINETUNED_MODEL_PATH)
base_from_adapter = peft_config.base_model_name_or_path
print('Base model recorded in adapter:', base_from_adapter)
print('Force base model:', peft_config.base_model_name_or_path)

# 베이스 모델 로드
base_model_name = base_from_adapter
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)

# LoRA adapter 적용
model = PeftModel.from_pretrained(
    base_model,
    FINETUNED_MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)

model = model.merge_and_unload()

model.eval()
print('Model loaded successfully.')
print('Adapter path:', FINETUNED_MODEL_PATH)
print('Base model used:', base_model_name)

Base model recorded in adapter: Salesforce/Llama-xLAM-2-8b-fc-r
Force base model: Salesforce/Llama-xLAM-2-8b-fc-r


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded successfully.
Adapter path: /content/drive/MyDrive/TANGO2 - SDF/모델/20260414_xLAM-2-8b-fc-r_10486-row_50percent-action_min/best_model
Base model used: Salesforce/Llama-xLAM-2-8b-fc-r


In [ ]:
#Init meta log
meta_log = []

In [ ]:
# ===== Prompt / parsing utils =====
def build_prompt(q_text: str) -> str:
    return (
        "You are given environment data in JSON format ('q').\n"
        "Predict motor control values and return ONLY a JSON object with keys w1 to w8.\n"
        "All values must be integers. Do not add extra text.\n\n"
        f"q: {q_text}\n\n"
        "Output example: {\"w1\":,\"w2\":,\"w3\":,\"w4\":,\"w5\":,\"w6\":,\"w7\":,\"w8\":}"
    )


def extract_json_obj(text: str):
    cands = re.findall(r'\{[\s\S]*?\}', text)
    for c in cands[::-1]:
        try:
            obj = json.loads(c)
            if all(k in obj for k in motor_cols):
                return {k: int(round(float(obj[k]))) for k in motor_cols}
        except Exception:
            try:
                obj = ast.literal_eval(c)
                if all(k in obj for k in motor_cols):
                    return {k: int(round(float(obj[k]))) for k in motor_cols}
            except Exception:
                pass
    return None

prompt_save_dict = {"prompt":[]}

def predict_one(q_text: str, max_new_tokens: int = 128):
    messages = [
        {'role': 'user', 'content': build_prompt(q_text)},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )
    prompt_save_dict["prompt"].append(prompt)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    _do_sample = False
    _repetition_penalty = 1.001
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=_do_sample,
            repetition_penalty = _repetition_penalty,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    meta_log.append(f"prompt : {prompt}")
    #Write parameters of 'model.generate' into meta_log
    meta_log.append(f"max_new_tokens: {max_new_tokens}")
    meta_log.append(f"do_sample: {_do_sample}")
    meta_log.append(f"temperature: {_temperature}")
    meta_log.append(f"repetition_penalty: {_repetition_penalty}")

    gen_ids = outputs[0][inputs['input_ids'].shape[1]:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    pred_obj = extract_json_obj(gen_text)

    if pred_obj is None:
        pred_obj = {k: -999 for k in motor_cols}

    return pred_obj, gen_text

In [ ]:
# 2. 데이터 불러오기
# CSV 파일 구조에 'q' 컬럼이 질문(q_text) 데이터를 담고 있다고 가정합니다.
df = pd.read_csv(EVAL_DATA_PATH)
print(f"Total samples to predict: {len(df)}")

# 결과를 담을 리스트
results = []
generated_texts = []

# 3. 반복문을 통한 추론 실행
# 'q'는 build_prompt에 들어갈 환경 데이터 컬럼명입니다. 파일 구성에 따라 수정하세요.
for index, row in tqdm(df.iterrows(), total=len(df), desc="Predicting"):
    q_content = str(row['q'])  # 질문 텍스트 추출

    # 이전에 정의한 predict_one 함수 호출
    pred_obj, gen_text = predict_one(q_content)

    # 결과 저장
    results.append(pred_obj)
    generated_texts.append(gen_text)

# 4. 결과 병합 및 파일 저장
# 추출된 w1~w8 데이터를 데이터프레임으로 변환
pred_df = pd.DataFrame(results)

# 원본 데이터에 예측 결과와 모델의 원본 답변(gen_text)을 붙입니다.
output_df = pd.concat([df, pred_df], axis=1)
output_df['generated_response'] = generated_texts

# CSV로 저장
output_df.to_csv(OUTPUT_FILE_PATH, index=False, encoding='utf-8-sig')

print(f"추론 완료! 결과가 다음 경로에 저장되었습니다: {OUTPUT_FILE_PATH}")

Total samples to predict: 1310


Predicting:   0%|          | 0/1310 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


NameError: name '_temperature' is not defined